# 04_208 · Auditoría común de etiquetas finas y transversales

Este cuaderno **no entrena** ni altera modelos. Evalúa todos los resultados disponibles sobre el mismo `test` 4:1 y deja explícito qué papel cumplen las anotaciones auxiliares.
Al arrancar verifica el bundle de Drive, recupera sólo artefactos ausentes y conserva cualquier conflicto local para revisión. Qwen plano se carga mediante la selección operativa de validation (época 3), no mediante el alias `best_adapter`.

- En `04_205`, las etiquetas finas y los flags se usan como **objetivos auxiliares predichos** durante el fine-tuning de Qwen.
- En `04_206`, sus logits predichos se reutilizan como características latentes; nunca se inyectan etiquetas gold en inferencia.
- En los modelos clásicos y Transformers restantes, el entrenamiento es `coarse-only`; las etiquetas finas y transversales se usan aquí para auditar subgrupos, errores y priorización de revisión humana.

Usar las etiquetas auxiliares gold como predictores produciría fuga de información, porque no estarán disponibles para chunks nuevos en producción.

In [ ]:
from pathlib import Path
import hashlib, importlib, json, subprocess, sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import display, Markdown, Image

# Verificación no destructiva de resultados generados en Colab. Sólo se copian
# archivos ausentes; cualquier conflicto se conserva localmente y se informa.
DRIVE_BUNDLE = Path(r'G:\My Drive\PLN_colab_04_artifacts')
RECOVERY_SCRIPT = ROOT / 'scripts_auxiliares' / 'recuperar_resultados_colab_04_20x.ps1'
SYNC_TARGETS = {
    '04_205 Qwen plano operativo': Path('resultados/metricas/qwen3_06b_lora_acoso_amenaza_4/evaluacion_test_modelo_seleccionado.json'),
    '04_206 Qwen jerárquico': Path('resultados/metricas/qwen_jerarquico_4/resultado.json'),
    '04_202 Transformer plano': Path('resultados/metricas/transformer_plano_4/resultado.json'),
    '04_203 Transformer cascada': Path('resultados/metricas/experimentos_jerarquicos_4/cascada_binaria_multietiqueta_4_seguros_ampliados/resultado.json'),
    '04_204 Transformer jerárquico': Path('resultados/metricas/experimentos_jerarquicos_4/transformer_jerarquico_multitarea_4/resultado.json'),
}

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''): digest.update(block)
    return digest.hexdigest()

def _sync_state(relative):
    local, remote = ROOT / relative, DRIVE_BUNDLE / relative
    if local.is_file() and remote.is_file(): return 'idéntico' if _sha256(local) == _sha256(remote) else 'conflicto (se conserva local)'
    if local.is_file(): return 'solo local'
    if remote.is_file(): return 'solo Drive'
    return 'ausente'

states_before = {name: _sync_state(path) for name, path in SYNC_TARGETS.items()}
recoverable = [name for name, state in states_before.items() if state == 'solo Drive']
if recoverable:
    if sys.platform != 'win32' or not RECOVERY_SCRIPT.is_file():
        raise RuntimeError('Hay resultados sólo en Drive, pero no está disponible el recuperador local.')
    common = ['powershell', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(RECOVERY_SCRIPT), '-Workspace', str(ROOT), '-DriveBundle', str(DRIVE_BUNDLE)]
    modes = []
    if any(name.startswith(('04_202', '04_203', '04_204')) for name in recoverable): modes.append('-ComparisonOnly')
    if any(name.startswith(('04_205', '04_206')) for name in recoverable): modes.append('-Qwen04_20XOnly')
    for mode in modes:
        recovery = subprocess.run([*common, mode], text=True, capture_output=True)
        if recovery.returncode != 0: raise RuntimeError(f'Falló la recuperación {mode}:\n' + (recovery.stderr or recovery.stdout))
        print(recovery.stdout.strip())
elif not DRIVE_BUNDLE.is_dir():
    print(f'Drive no está disponible en {DRIVE_BUNDLE}; se usarán resultados locales.')

deployment_required = [ROOT / 'modelos/transformer_plano_4/e5_small/best_checkpoint.pt', ROOT / 'modelos/transformer_plano_4/e5_small/tokenizer/tokenizer.json']
if any(not path.is_file() for path in deployment_required) and sys.platform == 'win32' and DRIVE_BUNDLE.is_dir():
    recovery = subprocess.run(['powershell', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(RECOVERY_SCRIPT), '-Workspace', str(ROOT), '-DriveBundle', str(DRIVE_BUNDLE), '-DeploymentOnly'], text=True, capture_output=True)
    if recovery.returncode != 0: raise RuntimeError('Falló la recuperación para el registro de 05:\n' + (recovery.stderr or recovery.stdout))
    print(recovery.stdout.strip())

sync_status = pd.DataFrame([{'cuaderno': name, 'antes': states_before[name], 'después': _sync_state(path)} for name, path in SYNC_TARGETS.items()])
display(Markdown('### Verificación Drive → workspace'))
display(sync_status)

from scripts_auxiliares import entrenar_qwen_acoso_amenaza as q4
from scripts_auxiliares import analizar_auxiliares_modelos_4 as audit4
from scripts_auxiliares import registro_modelos_produccion_4 as deploy4
# Evita reutilizar versiones antiguas conservadas por el kernel.
importlib.invalidate_caches()
q4 = importlib.reload(q4)
audit4 = importlib.reload(audit4)
deploy4 = importlib.reload(deploy4)
operational_qwen = q4.load_operational_evaluation(load_scores=False, require_test=True)
display({'qwen_checkpoint_operativo': operational_qwen['selected_epoch'], 'seleccion': str(q4.OPERATIONAL_SELECTION_PATH.relative_to(ROOT)), 'test_usado_para_seleccion': False})

## 1. Inventario antes de ejecutar

La celda siguiente permite saber qué experimentos ya terminaron. Los resultados ausentes quedan registrados; no se inventan ni se sustituyen por ejecuciones parciales.

In [ ]:
available, missing = audit4.collect_models()
deployment_registry = deploy4.load_registry(verify_hashes=True)
display({
    'modelos_disponibles': [m['label'] for m in available],
    'regimen_de_supervision': {m['label']: m['regime'] for m in available},
    'resultados_pendientes': missing,
    'modelos_publicados_para_05': {slot: value['label'] for slot, value in deployment_registry['models'].items()},
})

## 2. Ejecutar la auditoría reproducible

Para cada etiqueta fina se calcula recall de su categoría gruesa (o especificidad para variantes de `SEGURO`). Para cada flag transversal se mide qué proporción queda capturada al revisar el 10% y 20% de chunks más cercanos a los umbrales de decisión. Esta es una auditoría descriptiva común, no una prueba causal de que la supervisión auxiliar sea responsable de una diferencia.

In [ ]:
result = audit4.run_analysis(review_fractions=(0.10, 0.20))
display(result)

In [ ]:
import pandas as pd

fine = pd.read_csv(audit4.OUTPUT_DIR / 'desempeno_por_etiqueta_fina.csv')
flags = pd.read_csv(audit4.OUTPUT_DIR / 'captura_flags_por_incertidumbre.csv')
display(fine.sort_values(['fine_label', 'value'], ascending=[True, False]))
display(flags.sort_values(['review_fraction', 'flag', 'flag_capture'], ascending=[True, True, False]))

In [ ]:
for figure in (audit4.FIGURE_DIR / 'desempeno_fino.png', audit4.FIGURE_DIR / 'captura_flags.png'):
    if figure.exists():
        display(Image(filename=str(figure)))
display(Markdown(f'Informe reproducible: `{audit4.REPORT_PATH.relative_to(ROOT)}`'))

## Interpretación

La comparación principal entre modelos sigue siendo por las cuatro etiquetas gruesas y `SEGURO`. Esta auditoría responde dos preguntas complementarias: (1) si el desempeño se mantiene en cada fenómeno fino y (2) si los flags transversales se concentran entre los casos que el modelo enviaría a revisión. Para afirmar que la supervisión auxiliar mejora causalmente el modelo se requiere comparar una ablación Qwen idéntica con y sin las pérdidas auxiliares.